# Regime Yield-Curve Classifier

Assigns one of six yield-curve regimes to each trading day using nominal 2Y and 10Y Treasury yields.

**Outputs:** `data/interest_rate_regimes.xlsx`, exhibits under `Notebooks/output/regime/`.

Set `REFRESH = True` in the config cell to pull the latest FRED yields before a presentation refresh.


## Specification

**Inputs.** Daily 2Y and 10Y Treasury yields (FRED `DGS2`, `DGS10`). Spread $S_t = y^{(10)}_t - y^{(2)}_t$.

**Trend:** $d^{(i)}_t = \mathrm{SMA}_{21}(y^{(i)}) - \mathrm{SMA}_{200}(y^{(i)})$; $d^{(S)}_t = d^{(10)}_t - d^{(2)}_t$.

**Flags:** $u^{(i)}_t \in \{+1,0,-1\}$ with dead-zones $\tau^{(2)}=\tau^{(10)}=10$ bp, $\tau^{(S)}=5$ bp.

**Persistence:** commit after $k=5$ consecutive raw-label days.

See decision table in project README / prior version for six named regimes + Unclassified.


In [ ]:
from __future__ import annotations

import io
import ssl
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

REPO = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
NOTEBOOK_DIR = REPO / "Notebooks" if (REPO / "Notebooks").exists() else Path.cwd()
DATA = REPO / "data"
OUT_XLSX = DATA / "interest_rate_regimes.xlsx"
YIELD_CACHE = DATA / "treasury_yields_2y_10y.csv"
OUTPUT = NOTEBOOK_DIR / "output" / "regime"
OUTPUT.mkdir(parents=True, exist_ok=True)

SMA_FAST, SMA_SLOW = 21, 200
TAU_LEVEL = 0.10
TAU_SPREAD = 0.05
PERSIST_K = 5
REFRESH = False  # set True to pull latest FRED before deck refresh
ANALYSIS_START = "2001"

REGIME_ORDER = [
    "Bull Steepener", "Bull Flattener", "Bear Steepener", "Bear Flattener",
    "Steepener Twist", "Flattener Twist", "Unclassified",
]
REGIME_COLORS = {
    "Bull Steepener": "#2E7D32", "Bull Flattener": "#81C784",
    "Bear Steepener": "#C62828", "Bear Flattener": "#EF9A9A",
    "Steepener Twist": "#1565C0", "Flattener Twist": "#64B5F6",
    "Unclassified": "#BDBDBD",
}


In [ ]:
_SSL_CTX = ssl._create_unverified_context()


def _http_get(url: str, timeout: int = 60) -> str:
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=timeout, context=_SSL_CTX) as r:
        return r.read().decode()


def _fetch_fred(series_id: str) -> pd.Series:
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
    df = pd.read_csv(io.StringIO(_http_get(url)))
    df.columns = ["date", series_id]
    df["date"] = pd.to_datetime(df["date"])
    df[series_id] = pd.to_numeric(df[series_id], errors="coerce")
    return df.set_index("date")[series_id]


def _fetch_treasury(start_year: int = 1990, end_year: int | None = None) -> pd.DataFrame:
    end_year = end_year or pd.Timestamp.today().year
    base = (
        "https://home.treasury.gov/resource-center/data-chart-center/interest-rates/"
        "daily-treasury-rates.csv/{y}/all?type=daily_treasury_yield_curve"
        "&field_tdr_date_value={y}&page&_format=csv"
    )
    parts = []
    for yr in range(start_year, end_year + 1):
        try:
            txt = _http_get(base.format(y=yr), timeout=45)
            d = pd.read_csv(io.StringIO(txt))
            if "Date" not in d.columns or "2 Yr" not in d.columns or "10 Yr" not in d.columns:
                continue
            d = d[["Date", "2 Yr", "10 Yr"]].copy()
            d.columns = ["date", "y2", "y10"]
            d["date"] = pd.to_datetime(d["date"], errors="coerce")
            d["y2"] = pd.to_numeric(d["y2"], errors="coerce")
            d["y10"] = pd.to_numeric(d["y10"], errors="coerce")
            parts.append(d.dropna(subset=["date"]))
        except Exception:
            continue
    out = pd.concat(parts, ignore_index=True).drop_duplicates("date").sort_values("date")
    return out.set_index("date")[["y2", "y10"]]


def _load_yields() -> pd.DataFrame:
    try:
        y2 = _fetch_fred("DGS2")
        y10 = _fetch_fred("DGS10")
        return pd.concat({"y2": y2, "y10": y10}, axis=1).sort_index()
    except Exception as e:
        print(f"FRED unavailable ({e}); falling back to Treasury.gov (1990+).")
        return _fetch_treasury(1990)


if YIELD_CACHE.exists() and not REFRESH:
    yields = pd.read_csv(YIELD_CACHE, parse_dates=["date"]).set_index("date")
else:
    yields = _load_yields()
    yields.to_csv(YIELD_CACHE)

yields = yields.loc[yields[["y2", "y10"]].dropna().index.min():]
yields = yields.ffill().dropna(subset=["y2", "y10"])
yields["S"] = yields["y10"] - yields["y2"]
print(yields.index.min().date(), "->", yields.index.max().date(), "|", len(yields), "rows")
yields.head()


In [ ]:
def sign_with_tau(d: pd.Series | np.ndarray, tau: float) -> np.ndarray:
    d = np.asarray(d, dtype=float)
    return np.where(d > tau, 1, np.where(d < -tau, -1, 0))


def regime_label(a: int, b: int, c: int) -> str:
    if a == 0 or b == 0:
        return "Unclassified"
    if a == -1 and b == -1:
        return "Bull Steepener" if c == 1 else ("Bull Flattener" if c == -1 else "Unclassified")
    if a == 1 and b == 1:
        return "Bear Steepener" if c == 1 else ("Bear Flattener" if c == -1 else "Unclassified")
    if a == -1 and b == 1:
        return "Steepener Twist"
    if a == 1 and b == -1:
        return "Flattener Twist"
    return "Unclassified"


def trend_diff(s: pd.Series, *, fast: int, slow: int, smooth: str = "sma") -> pd.Series:
    if smooth == "sma":
        return s.rolling(fast, min_periods=fast).mean() - s.rolling(slow, min_periods=slow).mean()
    if smooth == "ema":
        return s.ewm(span=fast, min_periods=fast, adjust=False).mean() - s.ewm(span=slow, min_periods=slow, adjust=False).mean()
    raise ValueError(f"unknown smooth={smooth}")


def persist_labels(labels: pd.Series, k: int) -> pd.Series:
    out = labels.copy().astype(object)
    out[:] = None
    current, streak = None, 0
    for i, lab in enumerate(labels):
        if lab == current:
            streak += 1
        else:
            current, streak = lab, 1
        if streak >= k:
            out.iloc[i] = current
        elif i > 0:
            out.iloc[i] = out.iloc[i - 1]
    return out


def build_regime_panel(
    yld: pd.DataFrame,
    *,
    sma_fast: int = SMA_FAST,
    sma_slow: int = SMA_SLOW,
    tau_level: float = TAU_LEVEL,
    tau_spread: float = TAU_SPREAD,
    persist_k: int = PERSIST_K,
    smooth: str = "sma",
) -> pd.DataFrame:
    d2 = trend_diff(yld["y2"], fast=sma_fast, slow=sma_slow, smooth=smooth)
    d10 = trend_diff(yld["y10"], fast=sma_fast, slow=sma_slow, smooth=smooth)
    dS = d10 - d2
    u2 = sign_with_tau(d2, tau_level)
    u10 = sign_with_tau(d10, tau_level)
    uS = sign_with_tau(dS, tau_spread)
    uS_eff = uS.copy()
    uS_eff[(u2 == -1) & (u10 == 1)] = 1
    uS_eff[(u2 == 1) & (u10 == -1)] = -1
    raw = [
        regime_label(int(a), int(b), int(c))
        for a, b, c in zip(u2, u10, uS_eff)
    ]
    regime_raw = pd.Series(raw, index=yld.index, name="regime_raw")
    regime_filtered = persist_labels(regime_raw, persist_k)
    regime_filtered.name = "regime_filtered"
    return pd.DataFrame({
        "y2": yld["y2"], "y10": yld["y10"], "S": yld["S"],
        "d2": d2, "d10": d10, "dS": dS,
        "u2": u2, "u10": u10, "uS": uS_eff,
        "regime_raw": regime_raw, "regime_filtered": regime_filtered,
    }).dropna(subset=["d2", "d10"])


out = build_regime_panel(yields)
out.index.name = "date"
out.tail()


In [ ]:
labels_for_stats = out["regime_filtered"].dropna()
freq = labels_for_stats.loc["1990":].value_counts(normalize=True).sort_values(ascending=False)
runs = (labels_for_stats != labels_for_stats.shift()).cumsum()
block_lens = labels_for_stats.groupby(runs).size()
print("Regime frequencies (1990-present):")
print(freq.round(3))
print("\nMean filtered-block length:", round(block_lens.mean(), 1), "days")
print("Median filtered-block length:", int(block_lens.median()), "days")


In [ ]:
plot_df = out.loc["1990":].copy()
runs = (plot_df["regime_filtered"] != plot_df["regime_filtered"].shift()).cumsum()

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
ax_y = axes[0]
ax_y.plot(plot_df.index, plot_df["y2"], label="2Y", linewidth=0.9, color="#333333")
ax_y.plot(plot_df.index, plot_df["y10"], label="10Y", linewidth=0.9, color="#666666")
ax_y.set_ylabel("Yield (%)")
ax_y.legend(loc="upper right", fontsize=9)
ax_y.set_title("Treasury yields and filtered regime classification")

ax_r = axes[1]
for _, block in plot_df.groupby(runs):
    regime = block["regime_filtered"].iloc[0]
    ax_r.axvspan(block.index[0], block.index[-1], color=REGIME_COLORS.get(regime, "#CCC"), alpha=0.9, linewidth=0)
ax_r.set_yticks([])
handles = [mpatches.Patch(color=REGIME_COLORS[r], label=r) for r in REGIME_ORDER if r in plot_df["regime_filtered"].unique()]
ax_r.legend(handles=handles, loc="upper left", ncol=4, fontsize=8, frameon=False)
plt.tight_layout()
plt.savefig(OUTPUT / "regime_yields_timeline.png", bbox_inches="tight", dpi=150)
plt.show()


## Why ~50% Unclassified?

The PM question: what drives roughly half the sample being **Unclassified**? We decompose raw labels (before interpreting persistence) and show era-dependence.


In [ ]:
panel = out.loc[ANALYSIS_START:].copy()
raw = panel["regime_raw"]
u2, u10, uS = panel["u2"].astype(int), panel["u10"].astype(int), panel["uS"].astype(int)

def unclassified_reason(row) -> str:
    if row["regime_raw"] != "Unclassified":
        return row["regime_raw"]
    a, b, c = int(row["u2"]), int(row["u10"]), int(row["uS"])
    if a == 0:
        return "Unclassified: u2 dead-zone"
    if b == 0:
        return "Unclassified: u10 dead-zone"
    if c == 0:
        return "Unclassified: uS dead-zone"
    return "Unclassified: other"

panel["bucket"] = panel.apply(unclassified_reason, axis=1)
bucket_share = panel["bucket"].value_counts(normalize=True).sort_values(ascending=False)
attribution = bucket_share.reset_index()
attribution.columns = ["bucket", "share_all_days"]
attribution.to_csv(OUTPUT / "regime_unclassified_attribution.csv", index=False)
print("Day-type shares (2001+):")
print((bucket_share * 100).round(1).astype(str) + "%")

# waterfall chart: named vs three unclassified causes
named = [r for r in REGIME_ORDER if r != "Unclassified"]
cause_order = named + [
    "Unclassified: u2 dead-zone",
    "Unclassified: u10 dead-zone",
    "Unclassified: uS dead-zone",
]
plot_shares = bucket_share.reindex([c for c in cause_order if c in bucket_share.index]).fillna(0)

fig, ax = plt.subplots(figsize=(10, 5))
colors = [REGIME_COLORS.get(c.replace("Unclassified: ", ""), "#BDBDBD") if "Unclassified" not in c else "#BDBDBD" for c in plot_shares.index]
colors = [REGIME_COLORS.get(c, "#BDBDBD") if c in REGIME_COLORS else "#9E9E9E" for c in plot_shares.index]
ax.barh(plot_shares.index[::-1], (plot_shares.values * 100)[::-1], color=colors[::-1])
ax.set_xlabel("% of trading days (2001+)")
ax.set_title("Where each day lands: named regimes vs Unclassified causes")
plt.tight_layout()
plt.savefig(OUTPUT / "regime_unclassified_waterfall.png", bbox_inches="tight", dpi=150)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, tau, title in zip(
    axes,
    ["d2", "d10", "dS"],
    [TAU_LEVEL, TAU_LEVEL, TAU_SPREAD],
    ["|d2| (2Y trend)", "|d10| (10Y trend)", "|dS| (spread trend)"],
):
    x = panel[col].abs().dropna()
    ax.hist(x, bins=80, color="#546E7A", alpha=0.85, edgecolor="white", linewidth=0.3)
    ax.axvline(tau, color="#C62828", ls="--", lw=1.5, label=f"tau = {tau*100:.0f} bp")
    ax.set_title(title)
    ax.set_xlabel("abs differential")
    ax.legend(fontsize=8)
    ax.text(0.98, 0.95, f"P(|d|<=tau) = {(x <= tau).mean():.1%}", transform=ax.transAxes, ha="right", va="top", fontsize=9)
plt.suptitle("Trend differentials vs dead-zone (2001+)", y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT / "regime_diff_histograms.png", bbox_inches="tight", dpi=150)
plt.show()

annual_uncl = panel.groupby(panel.index.year)["regime_filtered"].apply(lambda s: (s == "Unclassified").mean())
roll_uncl = (panel["regime_filtered"] == "Unclassified").astype(float).rolling(252, min_periods=126).mean()

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=False)
axes[0].bar(annual_uncl.index.astype(str), annual_uncl.values * 100, color="#78909C")
axes[0].set_ylabel("% Unclassified (filtered)")
axes[0].set_title("Annual share of Unclassified — not constant (~5% in 2022, ~94% in 2021)")
axes[0].tick_params(axis="x", rotation=45)

axes[1].plot(roll_uncl.index, roll_uncl * 100, color="#37474F", lw=1)
axes[1].set_ylabel("% Unclassified (252d roll.)")
axes[1].set_title("Rolling 252-day Unclassified share (filtered)")
plt.tight_layout()
plt.savefig(OUTPUT / "regime_unclassified_over_time.png", bbox_inches="tight", dpi=150)
plt.show()


**Conclusion.** Roughly half of the 2001+ sample is **Unclassified by design**: the 21/200 SMA trend differentials often sit inside the ±10 bp (2Y/10Y) and ±5 bp (spread) dead-zones, so the classifier refuses to force a curve narrative in low-signal periods. Among Unclassified days, the 2Y dead-zone is the largest single cause; spread ambiguity matters when both yield legs agree in direction but |dS| ≤ τ.

The **5-day persistence filter does not create** the ~50% rate (raw vs filtered Unclassified shares are nearly identical). Era-dependence is large: violent rate moves (e.g. 2022) produce few Unclassified days; choppy/low-magnitude trends (e.g. 2021) produce many. This is conservative labeling, not a pipeline defect.


## Robustness — k, τ, and smoothing

Sensitivity on the 2001+ sample: which parameter moves the Unclassified share?


In [ ]:
def scenario_stats(**kwargs) -> dict:
    p = build_regime_panel(yields, **kwargs).loc[ANALYSIS_START:]
    raw = p["regime_raw"]
    filt = p["regime_filtered"].dropna()
    return {
        "pct_unclassified_raw": (raw == "Unclassified").mean(),
        "pct_unclassified_filtered": (filt == "Unclassified").mean(),
        "pct_named_filtered": (filt != "Unclassified").mean(),
        "bull_steepener": (filt == "Bull Steepener").mean(),
        "bear_flattener": (filt == "Bear Flattener").mean(),
        "median_block_days": int(
            filt.groupby((filt != filt.shift()).cumsum()).size().median()
        ),
    }

rows = []
for k in [1, 3, 5, 7, 10]:
    s = scenario_stats(persist_k=k)
    rows.append({"scenario": "persist_k", "param": k, **s})
for tau in [0.05, 0.10, 0.15, 0.20]:
    s = scenario_stats(tau_level=tau)
    rows.append({"scenario": "tau_level", "param": tau, **s})
for tau in [0.02, 0.05, 0.10, 0.15]:
    s = scenario_stats(tau_spread=tau)
    rows.append({"scenario": "tau_spread", "param": tau, **s})
for fast, slow, label in [(21, 200, "SMA 21/200"), (42, 252, "SMA 42/252"), (10, 100, "SMA 10/100")]:
    s = scenario_stats(sma_fast=fast, sma_slow=slow, smooth="sma")
    rows.append({"scenario": "smoothing", "param": label, **s})
s = scenario_stats(smooth="ema")
rows.append({"scenario": "smoothing", "param": "EMA 21/200", **s})

robust = pd.DataFrame(rows)
robust.to_csv(OUTPUT / "regime_robustness_grid.csv", index=False)
display_cols = ["scenario", "param", "pct_unclassified_filtered", "pct_named_filtered", "median_block_days"]
print(robust[display_cols].to_string(index=False, float_format=lambda x: f"{x:.3f}" if isinstance(x, float) else str(x)))

fig, ax = plt.subplots(figsize=(10, 5))
sub = robust[robust["scenario"].isin(["tau_level", "persist_k"])].copy()
sub["label"] = sub["scenario"] + "=" + sub["param"].astype(str)
colors = ["#C62828" if sc == "tau_level" else "#1565C0" for sc in sub["scenario"]]
ax.barh(sub["label"][::-1], (sub["pct_unclassified_filtered"] * 100)[::-1], color=colors[::-1])
ax.axvline(robust.loc[(robust.scenario == "persist_k") & (robust.param == 5), "pct_unclassified_filtered"].iloc[0] * 100,
           color="#333", ls=":", label="baseline k=5")
ax.set_xlabel("% Unclassified (filtered)")
ax.set_title("Robustness: tau_level moves the needle; persist_k does not")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT / "regime_robustness_bar.png", bbox_inches="tight", dpi=150)
plt.show()


In [ ]:
# Side-by-side filtered regimes: tighter vs baseline tau (2020+)
compare = pd.DataFrame({
    "tau_10bp": build_regime_panel(yields, tau_level=0.10)["regime_filtered"],
    "tau_5bp": build_regime_panel(yields, tau_level=0.05)["regime_filtered"],
}).loc["2020":].dropna()

fig, axes = plt.subplots(2, 1, figsize=(14, 3.5), sharex=True)
for ax, col in zip(axes, compare.columns):
    runs = (compare[col] != compare[col].shift()).cumsum()
    for _, block in compare.groupby(runs):
        reg = block[col].iloc[0]
        ax.axvspan(block.index[0], block.index[-1], color=REGIME_COLORS.get(reg, "#CCC"), alpha=0.9, linewidth=0)
    ax.set_yticks([])
    ax.set_ylabel(col)
axes[0].set_title("Lower tau labels more days — tradeoff: more false regime calls")
plt.tight_layout()
plt.savefig(OUTPUT / "regime_tau_compare_2020.png", bbox_inches="tight", dpi=150)
plt.show()


## Portfolio utility bridge

The classifier is useful for **portfolio management** when we connect curve states to how the commodity-trend sleeve behaves — not just regime frequencies.

Full conditional performance, event studies, and dynamic MV weights: re-run [`trend_vs_60_40_regime_analysis.ipynb`](trend_vs_60_40_regime_analysis.ipynb) if `interest_rate_regimes.xlsx` changes.


In [ ]:
# Current market snapshot
rf = out["regime_filtered"].dropna()
runs = (rf != rf.shift()).cumsum()
last_run_id = rf.groupby(runs).ngroup().iloc[-1]
block = out.loc[rf.groupby(runs).ngroup() == last_run_id]
last = out.dropna(subset=["regime_filtered"]).iloc[-1]

snapshot = pd.DataFrame([{
    "as_of": last.name.date(),
    "regime_filtered": last.regime_filtered,
    "regime_raw": last.regime_raw,
    "block_days": len(block),
    "block_start": block.index.min().date(),
    "y2_pct": round(last.y2, 2),
    "y10_pct": round(last.y10, 2),
    "spread_bp": round(last.S * 100, 1),
    "d2_bp": round(last.d2 * 100, 1),
    "d10_bp": round(last.d10 * 100, 1),
    "dS_bp": round(last.dS * 100, 1),
    "u2": int(last.u2),
    "u10": int(last.u10),
}])
print(snapshot.T)

q1_path = OUTPUT / "q1_trend_by_regime.csv"
if q1_path.exists():
    q1 = pd.read_csv(q1_path)
    q1 = q1.set_index("Regime")
    sharpe = q1["Sharpe"].reindex([r for r in REGIME_ORDER if r in q1.index])
    fig, ax = plt.subplots(figsize=(9, 4.5))
    bar_colors = [REGIME_COLORS.get(r, "#999") for r in sharpe.index]
    bars = ax.bar(sharpe.index, sharpe.values, color=bar_colors, edgecolor="white")
    if "Unclassified" in sharpe.index:
        bars[list(sharpe.index).index("Unclassified")].set_alpha(0.45)
    ax.axhline(0, color="#333", lw=0.8)
    ax.set_ylabel("Trend Sharpe (in-regime)")
    ax.set_title("Commodity-trend heterogeneity by yield-curve regime (Method 1, post-2001)")
    ax.tick_params(axis="x", rotation=35)
    u_sh = sharpe.get("Unclassified", np.nan)
    ax.text(0.02, 0.98, f"Unclassified still positive (Sharpe ~{u_sh:.2f}) but not a curve story",
            transform=ax.transAxes, va="top", fontsize=9)
    plt.tight_layout()
    plt.savefig(OUTPUT / "regime_trend_sharpe_by_regime.png", bbox_inches="tight", dpi=150)
    plt.show()
else:
    print("Run trend_vs_60_40_regime_analysis.ipynb first for q1_trend_by_regime.csv")


**Why allocators should care (May 2026):**

1. **Directional vs transition:** Named regimes = both yield legs trending beyond τ; Unclassified = curve dynamics not clear enough to label — still run the model, but lower conviction on regime-conditional tilts.
2. **Heterogeneous trend environments:** Bull Steepener historically strongest for the sleeve; Bear Flattener weakest — connects to the MV optimizer paradox (backward-looking weights underweight trend in the best regime).
3. **Current call:** Filtered regime and block length in the snapshot above; Bear Flattener = rising yields with spread compression, distinct from Bull Steepener (steepening from the short end).


In [ ]:
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as xw:
    out.reset_index().to_excel(xw, sheet_name="Regime Classification", index=False)
    pd.DataFrame({"regime": freq.index, "frequency_1990_present": freq.values}).to_excel(
        xw, sheet_name="Frequencies", index=False
    )
    attribution.to_excel(xw, sheet_name="Unclassified Attribution", index=False)
    robust.to_excel(xw, sheet_name="Robustness", index=False)
    snapshot.to_excel(xw, sheet_name="Current Snapshot", index=False)
print("Wrote:", OUT_XLSX.resolve())
print("Exhibits:", OUTPUT.resolve())
